In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from gensim.models import KeyedVectors
from tqdm import tqdm
import re


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/headline_data.csv")

texts = df["text"].astype(str).tolist()
headlines = df["headline"].astype(str).tolist()


In [ ]:
def clean_text(t):
    t = re.sub(r'[^\u1000-\u109F\s]', ' ', t)  # keep Myanmar chars
    return re.sub(r'\s+', ' ', t).strip()

texts = [clean_text(t) for t in texts]
headlines = [clean_text(h) for h in headlines]

def tokenize(t):
    return t.split()


In [ ]:
print("Loading fastText vectors...")
ft = KeyedVectors.load_word2vec_format("/content/drive/MyDrive/cc.my.300.vec")
embedding_dim = 300


In [ ]:
from collections import Counter

counter = Counter()

for t in texts + headlines:
    counter.update(tokenize(t))

vocab = ["<pad>", "<unk>", "<sos>", "<eos>"] + [w for w, c in counter.items() if c >= 2]

word2idx = {w:i for i, w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}
vocab_size = len(vocab)

print("Vocab size:", vocab_size)


In [ ]:
embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, embedding_dim))

for word, idx in word2idx.items():
    if word in ft:
        embedding_matrix[idx] = ft[word]

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float)


In [ ]:
MAX_TEXT_LEN = 200
MAX_HEAD_LEN = 20

def encode_sentence(sentence, max_len, add_sos_eos=False):
    tokens = tokenize(sentence)
    ids = []

    if add_sos_eos:
        ids.append(word2idx["<sos>"])

    for tok in tokens[:max_len]:
        ids.append(word2idx.get(tok, word2idx["<unk>"]))

    if add_sos_eos:
        ids.append(word2idx["<eos>"])

    if len(ids) < max_len:
        ids += [word2idx["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids


In [ ]:
class HeadlineDataset(Dataset):
    def __init__(self, texts, headlines):
        self.texts = texts
        self.headlines = headlines

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        src = encode_sentence(self.texts[idx], MAX_TEXT_LEN)
        trg = encode_sentence(self.headlines[idx], MAX_HEAD_LEN, add_sos_eos=True)
        return torch.tensor(src), torch.tensor(trg)

dataset = HeadlineDataset(texts, headlines)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        pred = self.fc(output.squeeze(1))
        return pred, hidden, cell


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output

            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

enc = Encoder(vocab_size, embedding_dim, 256, embedding_matrix)
dec = Decoder(vocab_size, embedding_dim, 256, embedding_matrix)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<pad>"])


In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for src, trg in tqdm(loader):
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()
        output = model(src, trg)

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")
